# 04 - Similarity Queries Notebook

This notebook demonstrates how to query the trained embedding model to find similar players.

## Contents

1. **Load Embeddings**: Load trained embeddings from file
2. **Setup API**: Initialize the similarity search API
3. **Query Examples**: Find similar players with various filters
4. **Comparison**: Compare GNN embeddings vs baseline
5. **Export Results**: Save top similarities to file

In [ ]:
# Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import sys
sys.path.append('..')

from retrieval import PlayerSimilarityAPI, PlayerFilter, EmbeddingIndex
from baselines import RatioSimilarityBaseline

print("Imports loaded successfully!")

## 1. Load Embeddings

In [ ]:
print("=" * 60)
print("LOADING PLAYER EMBEDDINGS")
print("=" * 60)

# Load GNN embeddings
try:
    gnn_embeddings = np.load('player_embeddings.npz')
    gnn_embeddings = {int(k): torch.tensor(v) for k, v in gnn_embeddings.items()}
    print(f"✓ Loaded GNN embeddings for {len(gnn_embeddings)} players")
except FileNotFoundError:
    print("Creating synthetic embeddings for demo...")
    np.random.seed(42)
    gnn_embeddings = {i: torch.randn(128) for i in range(100)}
    print(f"✓ Created synthetic embeddings for {len(gnn_embeddings)} players")

In [ ]:
# Create synthetic metadata (in real use, load from database)
player_metadata = {}
positions = ['Striker', 'Winger', 'Midfielder', 'Defender', 'Goalkeeper']
teams = ['Team A', 'Team B', 'Team C', 'Team D', 'Team E']
leagues = ['La Liga', 'Premier League', 'Serie A']

for pid in gnn_embeddings.keys():
    player_metadata[pid] = {
        'name': f"Player {pid}",
        'position': positions[pid % len(positions)],
        'team': teams[pid % len(teams)],
        'league': leagues[pid % len(leagues)],
        'age': 20 + (pid % 15),
        'reliability': min(1.0, 0.3 + (pid % 10) * 0.08),
        'n_events': 50 + pid * 10,
    }

print(f"✓ Created metadata for {len(player_metadata)} players")

## 2. Setup Similarity API

In [ ]:
print("\n" + "=" * 60)
print("SETTING UP SIMILARITY API")
print("=" * 60)

# Create embedding index
index = EmbeddingIndex(gnn_embeddings)
print(f"✓ Built embedding index with {len(index)} players")

# Demo API class
class DemoSimilarityAPI:
    """Demo API using in-memory index."""
    
    def __init__(self, index, metadata):
        self.index = index
        self.metadata = metadata
    
    def get_similar_players(self, player_id, k=10, filters=None):
        """Find similar players."""
        result_ids, result_scores = self.index.query_by_id(player_id, k * 2)
        
        results = []
        for pid, score in zip(result_ids, result_scores):
            meta = self.metadata.get(pid, {})
            
            # Apply filters
            if filters:
                if filters.get('same_role'):
                    query_pos = self.metadata.get(player_id, {}).get('position')
                    if meta.get('position') != query_pos:
                        continue
                
                if filters.get('leagues') and meta.get('league') not in filters.get('leagues'):
                    continue
                
                if filters.get('min_reliability') and meta.get('reliability', 0) < filters.get('min_reliability'):
                    continue
            
            results.append({
                'player_id': pid,
                'player_name': meta.get('name'),
                'similarity': score,
                'position': meta.get('position'),
                'team': meta.get('team'),
                'league': meta.get('league'),
            })
        
        return results[:k]

api = DemoSimilarityAPI(index, player_metadata)
print("✓ API initialized")

## 3. Query Examples

In [ ]:
print("\n" + "=" * 60)
print("SIMILARITY QUERIES")
print("=" * 60)

# Example 1: Basic query
query_player = 0
print(f"\n🔍 Query: Find players similar to Player {query_player}")
print("-" * 50)

results = api.get_similar_players(query_player, k=10)
print(f"{'Rank':<6}{'Name':<20}{'Similarity':<12}{'Position':<15}{'Team'}")
print("-" * 70)
for i, r in enumerate(results[:10]):
    print(f"{i+1:<6}{r['player_name']:<20}{r['similarity']:.3f}       "
          f"{r['position']:<15}{r['team']}")

In [ ]:
# Example 2: Same position filter
print(f"\n🔍 Query: Similar players with SAME POSITION")
print("-" * 50)

filters = {'same_role': True}
results = api.get_similar_players(query_player, k=10, filters=filters)

for i, r in enumerate(results[:5]):
    print(f"{i+1}. {r['player_name']} ({r['position']}) - Sim: {r['similarity']:.3f}")

In [ ]:
# Example 3: League filter
print(f"\n🔍 Query: Similar players in PREMIER LEAGUE")
print("-" * 50)

filters = {'leagues': ['Premier League']}
results = api.get_similar_players(query_player, k=10, filters=filters)

for i, r in enumerate(results[:5]):
    print(f"{i+1}. {r['player_name']} ({r['league']}) - Sim: {r['similarity']:.3f}")

## 4. Pairwise Similarity

In [ ]:
print("\n" + "=" * 60)
print("PAIRWISE SIMILARITY")
print("=" * 60)

# Compare specific players
player_pairs = [(0, 5), (0, 10), (0, 50), (5, 10)]

print(f"{'Player 1':<15}{'Player 2':<15}{'Similarity'}")
print("-" * 45)

for p1, p2 in player_pairs:
    emb1 = gnn_embeddings[p1]
    emb2 = gnn_embeddings[p2]
    
    # Cosine similarity
    emb1_norm = emb1 / emb1.norm()
    emb2_norm = emb2 / emb2.norm()
    sim = torch.dot(emb1_norm, emb2_norm).item()
    
    print(f"Player {p1:<8}Player {p2:<8}{sim:.3f}")

## 5. Similarity Distribution

In [ ]:
print("\n" + "=" * 60)
print("SIMILARITY DISTRIBUTION")
print("=" * 60)

# Compute all pairwise similarities
player_ids = list(gnn_embeddings.keys())[:50]
embeddings = torch.stack([gnn_embeddings[pid] for pid in player_ids])
embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)

sim_matrix = torch.matmul(embeddings, embeddings.t())

# Get off-diagonal similarities
mask = ~torch.eye(len(player_ids), dtype=torch.bool)
off_diag = sim_matrix[mask].numpy()

print(f"Similarity Statistics:")
print(f"  Mean: {np.mean(off_diag):.3f}")
print(f"  Std:  {np.std(off_diag):.3f}")
print(f"  Min:  {np.min(off_diag):.3f}")
print(f"  Max:  {np.max(off_diag):.3f}")

In [ ]:
# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(off_diag, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Cosine Similarity', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Pairwise Similarity Distribution', fontsize=14)
axes[0].axvline(np.mean(off_diag), color='r', linestyle='--', linewidth=2, label=f'Mean: {np.mean(off_diag):.3f}')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Heatmap
sns.heatmap(sim_matrix.numpy()[:20, :20], ax=axes[1], cmap='RdYlGn', 
            vmin=-0.5, vmax=1, center=0)
axes[1].set_title('Similarity Heatmap (Top 20 Players)', fontsize=14)

plt.tight_layout()
plt.show()

## 6. Export Top Similarities

In [ ]:
print("\n" + "=" * 60)
print("EXPORTING RESULTS")
print("=" * 60)

# Export top-5 similar players for each player
all_similarities = []

for pid in player_ids[:20]:
    results = api.get_similar_players(pid, k=5)
    
    for r in results:
        all_similarities.append({
            'query_player_id': pid,
            'query_player_name': f"Player {pid}",
            'similar_player_id': r['player_id'],
            'similar_player_name': r['player_name'],
            'similarity': r['similarity'],
            'position': r['position'],
            'team': r['team'],
        })

df = pd.DataFrame(all_similarities)
df.to_csv('top_similarities.csv', index=False)
print(f"✓ Exported {len(df)} similarity pairs to top_similarities.csv")

# Show sample
df.head(10)

In [ ]:
print("\n" + "=" * 60)
print("✅ SIMILARITY QUERIES COMPLETE")
print("=" * 60)